# Linear Regression Cross-Sectional Return Predictor

Walk-forward backtest using regularised linear regression (Ridge) on the full S&P 500 universe.
Results are saved to `results/linear_regression/`.

This notebook follows the exact same structure as `xgboost.ipynb` —
swap `LinearRegressionRankModel` for any other model that implements `AbstractCrossSectionalModel`.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '../../src')

from evaluation import WalkForwardBacktester
from models.base import AbstractCrossSectionalModel
import feature_engineer
import utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1  Model Definition

Define the Ridge regression model by implementing `AbstractCrossSectionalModel`.

In [4]:
from sklearn.linear_model import Ridge
from typing import Optional

class LinearRegressionRankModel(AbstractCrossSectionalModel):
    """Ridge regression trained on cross-sectional rank targets."""

    def __init__(self, alpha: float = 1.0) -> None:
        self._alpha = alpha
        self._model: Optional[Ridge] = None

    @property
    def name(self) -> str:
        return "linear_regression"

    def fit(self, X: pd.DataFrame, y: pd.Series) -> None:
        self._model = Ridge(alpha=self._alpha, fit_intercept=True)
        self._model.fit(X.fillna(X.median()), y)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if self._model is None:
            raise RuntimeError("Call fit() before predict().")
        return self._model.predict(X.fillna(X.median()))

    def get_feature_importance(self) -> Optional[pd.Series]:
        if self._model is None:
            return None
        return pd.Series(np.abs(self._model.coef_), index=self._feature_names).sort_values(ascending=False)

    def fit(self, X: pd.DataFrame, y: pd.Series) -> None:
        self._feature_names = list(X.columns)
        self._model = Ridge(alpha=self._alpha, fit_intercept=True)
        self._model.fit(X.fillna(X.median()), y)

    def get_params(self) -> dict:
        return {"alpha": self._alpha}

model = LinearRegressionRankModel(alpha=1.0)
print(f"Model: {model.name}  |  params: {model.get_params()}")

Model: linear_regression  |  params: {'alpha': 1.0}


## 2  Data Loading

In [6]:
df = pd.read_parquet("../../data/raw/yahoo_raw.parquet")
df = df.stack(level="Ticker", future_stack=True).reset_index()
df = df.sort_values(["Ticker", "Date"]).set_index("Date")
df.columns.name = None
print(f"Loaded {df['Ticker'].nunique()} tickers, {len(df):,} rows")

Loaded 503 tickers, 3,192,541 rows


## 3  Feature Engineering & Target

In [7]:
FEATURE_CONFIG = {
    "mom_windows":  [(1, 5), (1, 21), (21, 126), (21, 252), (126, 252), (252, 756)],
    "vol_windows":  [21, 63, 252],
    "liq_windows":  [21, 63],
    "high_windows": [252],
    "max_windows":  [21],
}
FEATURE_COLS = [
    "mom_1_5", "mom_1_21", "mom_21_126", "mom_21_252", "mom_126_252", "mom_252_756",
    "vol_21", "downside_dev_21", "vol_63", "downside_dev_63", "vol_252", "downside_dev_252",
    "dollar_volume_21", "dollar_volume_63", "dist_252_high", "max_21",
]

df = feature_engineer.compute_momentum_features(df, FEATURE_CONFIG["mom_windows"])
df = feature_engineer.compute_volatility_features(df, FEATURE_CONFIG["vol_windows"])
df = feature_engineer.compute_liquidity(df, FEATURE_CONFIG["liq_windows"])
df = feature_engineer.compute_extreme_features(
    df, high_windows=FEATURE_CONFIG["high_windows"], max_windows=FEATURE_CONFIG["max_windows"]
)

df["fwd_return_21"] = (
    df.groupby("Ticker")["Adj Close"].pct_change(21).shift(-21)
)
df["target"] = df.groupby("Date")["fwd_return_21"].rank(pct=True)
df[FEATURE_COLS] = df.groupby("Date")[FEATURE_COLS].rank(pct=True)

t0 = df.reset_index().groupby("Date")["mom_252_756"].count()
t0 = t0[t0 >= 1].index[0]
df = df[df.index > t0]
print(f"Panel: {df.shape}  |  {df.index.min().date()} → {df.index.max().date()}")

/Users/thiago/Documents/cross_sectional_return_prediction/notebooks/models/../../src/feature_engineer.py:23: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret = df.groupby("Ticker")["Adj Close"].pct_change()
/Users/thiago/Documents/cross_sectional_return_prediction/notebooks/models/../../src/feature_engineer.py:57: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret = g_price.pct_change()
/var/folders/5z/zvsvyk4s4hz0g367sh6yhx400000gn/T/ipykernel_47058/421825411.py:22: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in 

Panel: (2811770, 25)  |  2004-01-09 → 2026-03-30


## 4  Walk-Forward Backtest

In [ ]:
MODEL_NAME = "linear_regression"
RESULTS_DIR = "../../results/linear_regression"

backtester = WalkForwardBacktester(
    model=model,
    initial_train_months=84,
    test_months=12,
    step_months=1,
    embargo_days=21,
)

results = backtester.run(df, feature_cols=FEATURE_COLS, target_col="target")
print(f"Completed {len(results)} folds.")

Walk-forward [linear_regression]:  20%|██        | 37/183 [00:18<01:18,  1.87it/s]

## 5  Diagnostics

In [ ]:
ic_series, ls_spread = utils.diagnose(
    results,
    ic_on_every=21,
    save_dir=RESULTS_DIR,
    model_name=MODEL_NAME,
)

## 6  Feature Importance (Ridge Coefficients)

In [ ]:
importance = model.get_feature_importance()
if importance is not None:
    fig = utils.plot_feature_importance(
        importance, model_name=MODEL_NAME, top_n=len(FEATURE_COLS), save_dir=RESULTS_DIR
    )
    plt.show()